# House Prices: Advanced Regression Techniques
### Home Price Prediction (Kaggle Competition)
**Author:** Eduard Trubichkin  
**Date:** march 2026  
**Description:** Advanced boosting models (XGBoost, LightGBM, CatBoost) with feature engineering and hyperparameter optimization.

This notebook will describe the process of building models: XGBoost, LightGBM, CatBoost. But first, it is necessary to repeat all the actions that were performed in "01_house_prices_linear_models.ipynb".

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import Ridge, LassoCV, Lasso, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_squared_log_error

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# Filling in the gaps train
train['PoolQC'] = train['PoolQC'].fillna('NoPool')
train['MiscFeature'] = train['MiscFeature'].fillna('NoMiscFeature')
train['Alley'] = train['Alley'].fillna('NoAlley')
train['Fence'] = train['Fence'].fillna('NoFence')
train['MasVnrType'] = train['MasVnrType'].fillna('NoMasVnrType')
train['FireplaceQu'] = train['FireplaceQu'].fillna('NoFireplace')

train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

if train['LotFrontage'].isnull().any():
    global_median = train['LotFrontage'].median()
    train['LotFrontage'] = train['LotFrontage'].fillna(global_median)

train['GarageYrBlt'] = train['GarageYrBlt'].fillna(0)

garage_str_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_str_cols:
    train[col] = train[col].fillna('NoGarage')

bsmt_str_cols = ['BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1']
for col in bsmt_str_cols:
    train[col] = train[col].fillna('NoBasement')

train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
train.dropna(subset='Electrical', inplace=True)

train['SalePrice_log'] = np.log1p(train['SalePrice'])

#Filling in the gaps test
test['Alley'] = test['Alley'].fillna('NoAlley')
test['PoolQC'] = test['PoolQC'].fillna('NoPool')
test['MiscFeature'] = test['MiscFeature'].fillna('NoMiscFeature')
test['Fence'] = test['Fence'].fillna('NoFence')
test['MasVnrType'] = test['MasVnrType'].fillna('NoMasVnrType')
test['FireplaceQu'] = test['FireplaceQu'].fillna('NoFireplace')

garage_str = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_str:
    test[col] = test[col].fillna('NoGarage')

test['GarageYrBlt'] = test['GarageYrBlt'].fillna(0)
test['GarageArea'] = test['GarageArea'].fillna(0)
test['GarageCars'] = test['GarageCars'].fillna(0)

bsmt_str = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in bsmt_str:
    test[col] = test[col].fillna('NoBasement')

num_bsmt = ['BsmtFullBath', 'BsmtHalfBath', 'TotalBsmtSF', 'BsmtUnfSF', 'BsmtFinSF1', 'BsmtFinSF2']
for col in num_bsmt:
    test[col] = test[col].fillna(0)

lot_medians = train.groupby('Neighborhood')['LotFrontage'].median()
test['LotFrontage'] = test.apply(
    lambda row: lot_medians[row['Neighborhood']] if pd.isnull(row['LotFrontage']) else row['LotFrontage'],
    axis=1
)
global_median = train['LotFrontage'].median()
test['LotFrontage'] = test['LotFrontage'].fillna(global_median)

test['MasVnrArea'] = test['MasVnrArea'].fillna(0)

cat_cols = ['MSZoning', 'Functional', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'SaleType']
for col in cat_cols:
    if test[col].isnull().any():
        mode_val = train[col].mode()[0]
        test[col] = test[col].fillna(mode_val)


#feature engineering
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

train['HouseAge'] = train['YrSold'] - train['YearBuilt']
test['HouseAge'] = test['YrSold'] - test['YearBuilt']

train['RemodAge'] = train['YrSold'] - train['YearRemodAdd']
test['RemodAge'] = test['YrSold'] - test['YearRemodAdd']

train['HasGarage'] = (train['GarageArea'] > 0).astype(int)
test['HasGarage'] = (test['GarageArea'] > 0).astype(int)

train['HasBasement'] = (train['TotalBsmtSF'] > 0).astype(int)
test['HasBasement'] = (test['TotalBsmtSF'] > 0).astype(int)

train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
test['HasFireplace'] = (test['Fireplaces'] > 0).astype(int)

train['TotalBath'] = train['FullBath'] + 0.5*train['HalfBath'] + train['BsmtFullBath'] + 0.5*train['BsmtHalfBath']
test['TotalBath'] = test['FullBath'] + 0.5*test['HalfBath'] + test['BsmtFullBath'] + 0.5*test['BsmtHalfBath']

#Checking the size and absence of gaps
print(train.isnull().sum().sum())
print(test.isnull().sum().sum())
print(train.shape)
print(test.shape)

ordinal_mapping = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['NoBasement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['NoBasement', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['NoBasement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['NoFireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['NoGarage', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NoGarage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['NoPool', 'Fa', 'TA', 'Gd', 'Ex'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'PavedDrive': ['N', 'P', 'Y'],
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'Fence': ['NoFence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']
}

numeric_features = train.select_dtypes(include=['int64', 'float64']).columns.to_list()
numeric_features = [col for col in numeric_features if col not in ['Id', 'SalePrice', 'SalePrice_log']]

ordinal_features = list(ordinal_mapping.keys())

categorical_features = train.select_dtypes(include=['str']).columns.to_list()
categorical_features = [col for col in categorical_features if col not in ordinal_features]

categories_list = [ordinal_mapping[col] for col in ordinal_features]


ordinal_encoder = OrdinalEncoder(
    categories=categories_list,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

categorical_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)


preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('ord', ordinal_encoder, ordinal_features),
        ('cat', categorical_encoder, categorical_features)
    ]
)


X = train.drop(['Id', 'SalePrice', 'SalePrice_log'], axis=1)
Y = train['SalePrice_log']

X_processed = preprocessor.fit_transform(X)

X_test = test.drop(['Id'], axis=1)

X_test_processed = preprocessor.transform(X_test)

print(X_processed.shape)
print(X_test_processed.shape)

Now let's split train into validation and training datasets.

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_processed, Y, train_size=0.8, random_state=42
)

Now let's start implementing the XGBoost model. First, we implement the basic (initial) model. It is needed in order to use `early_stopping_rounds` to determine the optimal number of trees. Then we will search for the best hyperparameters through `RandomizedSearchCV`.   

This model should not be confused with a radiant descent. Gradient descent is an optimization technique for finding the minimum of a function. In linear regression, we use it to find the weights $w$ and the bias $b$ that minimize the loss function.   
Gradient boosting is a machine learning algorithm that builds an ensemble of decision trees (not a linear model). Each subsequent tree is trained to correct the mistakes of the previous ones. The name "gradient" comes from the fact that the idea of gradient descent is used inside the algorithm.: we are looking for a direction in which to adjust the predictions to reduce the error. But the weights themselves here are not coefficients for signs, but the structure of the trees. Thus, gradient boosting is a completely different model, not related to linear ones.

A decision tree is a model that divides the feature space into regions and predicts a constant in each region (for example, the average price in this region). Splits are performed by feature values, forming a tree: at each node, some feature is checked, and depending on the result, we go to the left or right branch.   
The depth of the tree (max_depth) is the maximum number of such partitions from the root to the leaf. Deep trees can learn complex patterns, but they are easily retrained (they remember noise, not the general trend). Shallow trees are simpler and more stable.   

Overfitting is a situation where a model adjusts too precisely to the training data, learning random noises and outliers, among other things, and loses its ability to generalize to new data. In boosting, a large number of trees or their structure is too deep leads to overfitting. To combat this, use:  
- depth limit (max_depth)
- subdescription (subsample, calsample_bytree)
- an early stop (early_stopping_rounds)

In the code, we first create an xgb_base model with a large number of trees (n_estimators=1000). When training with the parameter `early_stopping_rounds=50`, XGBoost monitors the quality in the validation sample. If the quality does not improve within 50 iterations, training stops. The `best_iteration' attribute stores the number of trees with the best quality. This allows you to automatically select the optimal number of trees without going through all 1000 manually.  
Thus, the basic model is needed in order to find out how many trees are actually required before retraining begins. Then we use this number in the final model (or in the selection of hyperparameters).

The gradient boosting model is implemented using the 'XGBRegressor` class. We'll initialize the object in parentheses.:  
- `n_estimators` - the maximum number of trees (boosting iterations)
- `learning_rate` - the step of updating weights (learning rate). A lower value requires more trees, but often gives a more stable result. 
- `max_depth` is the maximum depth of each tree. Deep trees can retrain, "5" is a good compromise
- `subsample` - the proportion of rows (objects) used to build each tree. It helps to combat overfitting. 
- `colsample_bytree` - the proportion of features (columns) used to build each tree. Similar to subsample, but based on
- `random_state` - captures random processes for reproducibility
- `n_jobs = -1` - use all available processor cores to accelerate
- `early_stopping_rounds` - the number of iterations after which the model training stops if there are no improvements.

In [ ]:
xgb_base = xgb.XGBRegressor(
    n_estimators = 1000,
    learning_rate = 0.05,
    max_depth = 5,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42,
    n_jobs = -1,
    early_stopping_rounds=50
)

Now let's train the model. As mentioned earlier, the basic model is needed to find the optimal number of trees - the number of iterations after which the quality in the validation sample stops improving. This allows you to stop before the start of retraining. And in the `fit()' method`the argument `early_stopping_rounds=50` is just being passed, which says: "if the validation metric has not improved during 50 consecutive iterations (of the trees), training stops." After that, 'xgb_base.best_iteration` will save the number of trees where the quality was the best. This number can be used for the final model.  

In general, the following arguments are passed to `fit()`:
- `X_train`, `Y_train' - training dataset
- `eval_set` - a list of tuples with validation data on which the metric will be monitored
- `verbose` - responsible for outputting logs to the console, if we want to see the process, set the value to "1", if not, set the value to "False"

In [ ]:
xgb_base.fit(
    X_train, Y_train,
    eval_set = [(X_val, Y_val)],
    verbose = False
)

print(f"Best n_estimators: {xgb_base.best_iteration}")
print(f"RMSLE: {np.sqrt(mean_squared_log_error(np.expm1(Y_val), np.expm1(xgb_base.predict(X_val)))):.5f}")

These are the results that were obtained:   
Best n_estimators: 281   
RMSLE: 0.1261   

Now let's move on to the selection of hyperparameters. First, we will create a dictionary with the parameters that we will select and the possible values.

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 250, 270, 300, 400],
    'learning_rate': np.linspace(0.01, 0.1, 10),
    'max_depth': [3, 5, 7, 9],
    'subsample' : [0.6, 0.8, 1],
    'colsample_bytree': [0.6, 0.8, 1],
    'reg_alpha': np.linspace(0, 0.5, 10),
    'reg_lambda': [0.5, 1, 1.5, 2]
}

In the dictionary, you need to pay attention to the last 2 keys. `reg_alpha` is L1 regularization (a penalty on absolute weights) that helps reduce the number of significant features. `reg_lambda` - L2 regularization (penalty on squared weights). It is usually used for smoothing.   

Now let's move on to creating a second XGBossting model and creating a RandomizedSearchCV object to select the best parameters.

In [ ]:
xgb_model = xgb.XGBRegressor(random_state = 42, n_jobs = -1, objective = 'reg:squarederror')
random_search = RandomizedSearchCV(
    xgb_model, param_dist, n_iter = 50, cv = 3, scoring = 'neg_mean_squared_error', 
    random_state = 42, n_jobs = -1, verbose = 1 
)
random_search.fit(X_train, Y_train)

print(f"Best parametrs: {random_search.best_params_}")
print(f"Best CV RMSE (in logarithnic scale): {np.sqrt(-random_search.best_score_):.4f}")

In the object whose function is to determine the optimal values, we pass the following arguments:
- `xgb_model` - the model for which we select the parameters
- `param_dist` - a dictionary with parameters and possible values to be sorted through
- `n_iter` - the number of random combinations of hyperparameters to be checked. Instead of going through all the possible options (of which there may be thousands), we randomly select 50 combinations. This speeds up the search. The larger the n_iter, the higher the chance of finding good parameters, but the longer the search takes.
- `cv` - number of folds during cross validation
- `scoring` is a metric by which the search compares different combinations of hyperparameters. Cross-validation is performed inside the search, and the value of this metric is calculated for each combination. Since "RandomizedSearchCV" maximizes the value (the more the better), we use a negative "MSE" (neg_mean_squared_error) so that a large negative "MSE" corresponds to a smaller positive "MSE".
- `random_state` - captures random processes for reproducibility
- `n_jobs` - use all available processor cores to accelerate
- `verbose` - responsible for outputting logs to the console. The value "1" indicates that the process will be visible in the console.

The training is based on a training sample (80%). 

After completing the training (`random_search.fit()`) will become available:  
- `random_search.best_params__` - dictionary with the best hyperparameters
- `random_search.best_score_` is the best metric value (negative "MSE"). To get a positive "MSE", you need to take "-random_search.best_score_"
- `random_search.best_estimator_` — a model trained on all data (within cross-validation) with the best parameters. You can use it for predictions, but it's better to retrain on everything "X_processed" with "best_params__" for purity. 

As a result, the basic model of XGBoost gave a better result on validation than the selection of parameters. Therefore, let's try to generate a final submit based on the basic model.

In [ ]:
preds_log = xgb_base.predict(X_test_processed)
preds_orig = np.expm1(preds_log)

submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': preds_orig
})

submission.to_csv('submission_xgboost_base.csv', index=False)

The basic XGBoost model showed an increase, albeit a small one. The current record is `Score = 0.13093`.